In [ ]:
import pandas as pd
import numpy as np


In [ ]:
sales_long = pd.read_csv(
    "../data/processed/model_data.csv",
    parse_dates=["date"]
)

sales_long.info()
sales_long.head()

In [ ]:
sales_long.isnull().sum().sort_values(ascending=False).head(10)

We know that weekends, month start or ends can affect the sales, also we have to know the weekly trends and quarterly trends of the sale in the store


In [ ]:
sales_long['day'] = sales_long['date'].dt.day

sales_long['week'] = sales_long['date'].dt.isocalendar().week.astype("int16")

sales_long["quarter"] = sales_long['date'].dt.quarter

sales_long['is_weekend'] = (
    sales_long["weekday"]
    .isin(["Saturday", "Sunday"])
    .astype("int8")
)

sales_long[
    [
        "date",
        "day",
        "week",
        "month",
        "quarter",
        "weekday",
        "is_weekend",
    ]
].head(10)

In [ ]:
# Calendar-derived features (if not already extracted into numeric form):
# day_of_week
# week_of_year
# is_weekend
# quarter
# cyclic encoding (optional)
# Event features (event_name_*, event_type_*)
# SNAP features (snap_CA, snap_TX, snap_WI)
# Price-derived features (price change, normalized price, etc.)
# Preparing the final feature matrix
# Train/validation split for time series
# Baseline forecasting model
# Model evaluation
# Inventory optimization layer (safety stock, reorder point, EOQ)



Working on Price features to make the model learn the price sales relation


Handling the Mising Values

In [ ]:
missing = sales_long.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing)

missing_percentage = (
    sales_long
    .isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_percentage[missing_percentage > 0]

In [ ]:
# so filling the event columns with "No Event" where there is no event present so to minimise the number of null values in them
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

sales_long[event_cols] = sales_long[event_cols].fillna("No_Event")

In [ ]:
print(sales_long["sell_price"].isna().mean())


items_with_missing_price = sales_long.loc[
    sales_long["sell_price"].isna(),
    "item_id"
].unique()

print(f"Number of items: {len(items_with_missing_price)}")
print(items_with_missing_price)


sales_long.loc[
    sales_long["sell_price"].isna(),
    ["item_id", "date", "wm_yr_wk"]
]



In [ ]:
sales_long[["item_id", "date", "wm_yr_wk", "sell_price"]]

In [ ]:
# to confirm whether the missing prices are due to introducing the item early on and not opening it for sale
# to test this hypothesis, first I did a try to find whether this item has a price at all by checking its tail of last 20 sales days it made
# then checking whether the item was sold on a day but the sell_price is not filled
# if they all are true then our assumption is correct
# checking for the product "FOODS_1_004"
print(sales_long.loc[
    sales_long["item_id"] == "FOODS_1_004",
    ["date", "sell_price"]
].tail(20))

print(sales_long.loc[
    (sales_long["item_id"] == "FOODS_1_004") &
    (sales_long["sell_price"].isna()),
    ["date", "sales"]
])

sales_long.loc[
    (sales_long["item_id"] == "FOODS_1_004") &
    (sales_long["sell_price"].isna()) &
    (sales_long["sales"] > 0),
    ["date", "sales"]
]

In [ ]:
sales_long.groupby("item_id")["sell_price"].apply(lambda x: x.notna().sum()).describe()

In [ ]:
# Here I am checking whether the rows having sales_price = 0 have sales > 0
# if not then we can just remove all the rows having sale_price = 0

print(sales_long.loc[
    sales_long["sell_price"].isna(),
    "sales"
].describe())

print(sales_long.loc[
    sales_long["sell_price"].isna(),
    "sales"
].value_counts().head(10))

In [ ]:
# Dropping the rows where sell_price = 0 as there is no benefit of having info about the product which is either not ready for sale at that time or was stopped selling by the store and so sales are also 0 for it that time
# Our model will not learn anything from it and filling these values also give false info to our model which is bad

sales_long = sales_long[sales_long["sell_price"].notna()].copy()
sales_long.reset_index(drop=True, inplace=True)

In [ ]:
sales_long["lag_1"] = (
    sales_long.groupby("item_id")["sales"]
    .shift(1)
)

sales_long["lag_7"] = (
    sales_long.groupby("item_id")["sales"]
    .shift(7)
)

sales_long["lag_28"] = (
    sales_long.groupby("item_id")["sales"]
    .shift(28)
)

In [ ]:
sales_long["rolling_mean_7"] = (
    sales_long.groupby("item_id")["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

sales_long["rolling_mean_28"] = (
    sales_long.groupby("item_id")["sales"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

sales_long["rolling_std_7"] = (
    sales_long.groupby("item_id")["sales"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

In [ ]:
sales_long["price_change"] = (
    sales_long.groupby("item_id")["sell_price"]
    .pct_change()
)

sales_long["price_max"] = (
    sales_long.groupby("item_id")["sell_price"]
    .transform("max")
)

sales_long["price_min"] = (
    sales_long.groupby("item_id")["sell_price"]
    .transform("min")
)

sales_long["price_norm"] = (
    sales_long["sell_price"] / sales_long["price_max"]
)

sales_long["price_range"] = (
    sales_long["price_max"] - sales_long["price_min"]
)

In [ ]:
missing = (
    sales_long.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

In [ ]:
# filling the price_change NA as 0 because they tell there is no last price to compare with, so puting 0 can work here

sales_long["price_change"] = sales_long["price_change"].fillna(0)


# dropping the rows where lag, rolling features have value NA
model_data = sales_long.dropna().reset_index(drop=True)

In [ ]:
missing = (
    model_data.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]